In [1]:
import pandas as pd
import numpy as np

In [ ]:
colunas = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT', 'NU_NOTA_REDACAO',
           'TX_RESPOSTAS_CN','TX_RESPOSTAS_CH','TX_RESPOSTAS_LC','TX_RESPOSTAS_MT',
           'TX_GABARITO_CN','TX_GABARITO_CH','TX_GABARITO_LC','TX_GABARITO_MT',
           'TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT',
           'CO_PROVA_CN','CO_PROVA_CH','CO_PROVA_LC','CO_PROVA_MT',
           'TP_LINGUA','NU_ANO']

df = pd.read_parquet("enem_parquet/2021", columns = colunas)

In [ ]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

In [ ]:
df = df[df['FALTOU'] == 0]

In [ ]:
print(df.isna().sum())

In [ ]:
def gerar_matriz(respostas, gabaritos):
    r = np.array([list(x) for x in respostas])
    g = np.array([list(x) for x in gabaritos])
    return (r == g).astype(np.int8)

mask_ing = df['TP_LINGUA'].values == 1

resp_lc = np.where(
    mask_ing,
    df['TX_RESPOSTAS_LC'].str[5:10] + df['TX_RESPOSTAS_LC'].str[10:],  # Inglês: posições 5-9
    df['TX_RESPOSTAS_LC'].str[0:5]  + df['TX_RESPOSTAS_LC'].str[10:]   # Espanhol: posições 0-4
)
gab_lc = np.where(
    mask_ing,
    df['TX_GABARITO_LC'].str[5:10] + df['TX_GABARITO_LC'].str[10:],
    df['TX_GABARITO_LC'].str[0:5]  + df['TX_GABARITO_LC'].str[10:]
)


AREAS = {
    'LC': (resp_lc, gab_lc),
    'CH': (df['TX_RESPOSTAS_CH'].values, df['TX_GABARITO_CH'].values),
    'CN': (df['TX_RESPOSTAS_CN'].values, df['TX_GABARITO_CN'].values),
    'MT': (df['TX_RESPOSTAS_MT'].values, df['TX_GABARITO_MT'].values)
   
}

matrizes = []
nomes_colunas = []

for sigla, (resp, gab) in AREAS.items():
    M = gerar_matriz(resp, gab)
    matrizes.append(M)
    nomes_colunas.extend([f'questao_{i+1}_{sigla}' for i in range(45)])

X = np.hstack(matrizes)
df_questoes = pd.DataFrame(X, columns=nomes_colunas, index=df.index)
df = pd.concat([df_questoes, df], axis=1)

In [ ]:
df_questoes

In [ ]:
df.to_parquet("dados_acertos_2021.parquet")